In [ ]:

# ---------------------------------------------------------------------
# CONFIGURAÇÕES INICIAIS DAS ANÁLISES
# ---------------------------------------------------------------------

import csv
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window


# Caminhos
SCRIPT_DIR = Path(__file__).resolve().parent
PROJECT_ROOT = Path(__file__).resolve().parents[3]

GOLD_06_DIR = PROJECT_ROOT / "Gold" / "perguntas_negocio" / "gold_06_diferencas_regiao_senioridade_modelo"
OUTPUT_DIR = SCRIPT_DIR.parent / "outputs" / "gold_06"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# Spark
spark = (
    SparkSession.builder
    .appName("TechChallenge_Analytics")
    .master("local[*]")
    .getOrCreate()
)


# ---------------------------------------------------------------------
# CARREGAR GOLD 06
"""
A análise utiliza somente os registros de faixa salarial, pois a pergunta de negócio compara diferenças de remuneração entre região, senioridade e modelo de trabalho. Manter uma única variável evita misturar distribuições de naturezas diferentes.
"""
# ---------------------------------------------------------------------

arquivos_gold_06 = [
    str(arquivo) for arquivo in GOLD_06_DIR.glob("part-*.csv")
]

print("\nGOLD 06:")
print(GOLD_06_DIR)

print("\nARQUIVOS ENCONTRADOS:")
print(arquivos_gold_06)

if not arquivos_gold_06:
    raise FileNotFoundError(
        f"Nenhum arquivo part-*.csv encontrado em: {GOLD_06_DIR}"
    )

df_gold_06 = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(arquivos_gold_06)
)

df = (
    df_gold_06
    .filter(F.col("variavel") == "faixa_salarial")
)


# ---------------------------------------------------------------------
# ORDENAR FAIXAS SALARIAIS
"""
Como a remuneração está registrada em faixas textuais, cada intervalo recebe uma ordem crescente. Essa ordenação permite calcular distribuição acumulada, P50, P75 e ranking sem transformar as faixas em salários pontuais.
"""
# ---------------------------------------------------------------------

df = (
    df.withColumn(
        "ordem_faixa",
        F.when(F.col("valor") == "Menos de R$ 1.000/mês", 1)
        .when(F.col("valor") == "de R$ 1.001/mês a R$ 2.000/mês", 2)
        .when(F.col("valor") == "de R$ 2.001/mês a R$ 3.000/mês", 3)
        .when(F.col("valor") == "de R$ 3.001/mês a R$ 4.000/mês", 4)
        .when(F.col("valor") == "de R$ 4.001/mês a R$ 6.000/mês", 5)
        .when(F.col("valor") == "de R$ 6.001/mês a R$ 8.000/mês", 6)
        .when(F.col("valor") == "de R$ 8.001/mês a R$ 12.000/mês", 7)
        .when(F.col("valor") == "de R$ 12.001/mês a R$ 16.000/mês", 8)
        .when(F.col("valor") == "de R$ 16.001/mês a R$ 20.000/mês", 9)
        .when(F.col("valor") == "de R$ 20.001/mês a R$ 25.000/mês", 10)
        .when(F.col("valor") == "de R$ 25.001/mês a R$ 30.000/mês", 11)
        .when(F.col("valor") == "de R$ 30.001/mês a R$ 40.000/mês", 12)
        .when(F.col("valor") == "Acima de R$ 40.001/mês", 13)
    )
)


# ---------------------------------------------------------------------
# VALIDAR MAPEAMENTO DAS FAIXAS
"""
A validação garante que todas as faixas utilizadas na análise receberam uma posição na escala salarial. No notebook executado, nenhuma faixa ficou sem classificação, portanto todos os registros puderam seguir para os cálculos de percentis.
"""
# ---------------------------------------------------------------------

print("\n" + "=" * 120)
print("VALIDAÇÃO DAS FAIXAS SALARIAIS")
print("=" * 120)

faixas_sem_ordem = (
    df
    .filter(F.col("ordem_faixa").isNull())
    .select("valor")
    .distinct()
)

if faixas_sem_ordem.count() > 0:
    print("\nATENÇÃO: existem faixas salariais sem classificação:")

    faixas_sem_ordem.show(100, truncate=False)

    raise ValueError(
        "Existem faixas salariais sem ordem definida."
    )

else:
    print(
        "\nTodas as faixas salariais foram classificadas corretamente."
    )


# ---------------------------------------------------------------------
# SIMPLIFICAR MODELO DE TRABALHO
"""
Os rótulos de modelo de trabalho são simplificados apenas para apresentação. As quatro categorias mantêm o mesmo significado, mas passam a ter nomes mais curtos nos resultados e arquivos exportados.
"""
# ---------------------------------------------------------------------

df = (
    df.withColumn(
        "modelo_trabalho",
        F.when(
            F.col("modelo_de_trabalho_atual") == "Modelo 100% remoto",
            "100% remoto"
        )
        .when(
            F.col("modelo_de_trabalho_atual") == "Modelo 100% presencial",
            "100% presencial"
        )
        .when(
            F.col("modelo_de_trabalho_atual")
            == "Modelo híbrido com dias fixos de trabalho presencial",
            "Híbrido com dias fixos"
        )
        .when(
            F.col("modelo_de_trabalho_atual")
            == "Modelo híbrido flexível (o funcionário tem liberdade para escolher quando estar no escritório presencialmente)",
            "Híbrido flexível"
        )
        .otherwise(F.col("modelo_de_trabalho_atual"))
    )
)


# ---------------------------------------------------------------------
# FUNÇÃO PARA EXPORTAR CSV
# ---------------------------------------------------------------------

def exportar_csv(
    df_exportar,
    nome_arquivo
):
    caminho = OUTPUT_DIR / f"{nome_arquivo}.csv"

    linhas = df_exportar.collect()

    with open(
        caminho,
        "w",
        newline="",
        encoding="utf-8-sig"
    ) as arquivo:
        writer = csv.writer(
            arquivo,
            delimiter=";"
        )

        writer.writerow(df_exportar.columns)

        for linha in linhas:
            writer.writerow(
                [
                    linha[coluna]
                    for coluna in df_exportar.columns
                ]
            )

    print(f"\nArquivo salvo: {caminho}")


# ---------------------------------------------------------------------
# VALIDAÇÃO DOS DENOMINADORES
"""
Antes dos percentis, é validado se a soma das faixas salariais coincide com o total informado pela Gold para cada combinação de edição, região, nível e modelo de trabalho. Também é verificado se existe apenas um denominador por combinação. A execução não encontrou inconsistências.
"""
# ---------------------------------------------------------------------

print("\n" + "=" * 120)
print("1. VALIDAÇÃO DOS DENOMINADORES")
print("=" * 120)

validacao_denominador = (
    df
    .groupBy(
        "edicao",
        "regiao_onde_mora",
        "nivel",
        "modelo_trabalho"
    )
    .agg(
        F.sum("contagem").alias("soma_contagem"),
        F.max("total_respondentes").alias("total_gold"),
        F.countDistinct("total_respondentes").alias("qtd_totais_gold")
    )
    .withColumn(
        "validacao",
        F.when(
            (
                F.col("soma_contagem")
                == F.col("total_gold")
            )
            &
            (
                F.col("qtd_totais_gold")
                == 1
            ),
            "OK"
        )
        .otherwise("VERIFICAR")
    )
)

inconsistencias = (
    validacao_denominador
    .filter(F.col("validacao") != "OK")
)

qtd_inconsistencias = (
    inconsistencias
    .count()
)

print(
    f"\nQuantidade de combinações com inconsistência: "
    f"{qtd_inconsistencias}"
)

if qtd_inconsistencias > 0:
    inconsistencias.show(100, truncate=False)

else:
    print("Denominadores validados.")


# ---------------------------------------------------------------------
# RESPONDENTES POR EDIÇÃO
"""
A base salarial utilizada reúne 3.723 respondentes em 2024-2025 e 2.428 em 2025-2026. Como o volume varia entre as edições, as comparações posteriores priorizam a posição das faixas salariais em vez de contagens absolutas.
"""
# ---------------------------------------------------------------------

print("\n" + "=" * 120)
print("2. RESPONDENTES POR EDIÇÃO")
print("=" * 120)

respondentes_edicao = (
    df
    .groupBy("edicao")
    .agg(
        F.sum("contagem").alias("total_respondentes")
    )
    .orderBy("edicao")
)

respondentes_edicao.show(truncate=False)


# ---------------------------------------------------------------------
# MAPA DAS FAIXAS SALARIAIS
"""
O mapa converte novamente a posição numérica para o rótulo original da faixa salarial após os cálculos. Assim, a ordenação funciona apenas como apoio analítico e os resultados continuam apresentados nas categorias originais da pesquisa.
"""
# ---------------------------------------------------------------------

FAIXAS_SALARIAIS = {
    1: "Menos de R$ 1.000/mês",
    2: "de R$ 1.001/mês a R$ 2.000/mês",
    3: "de R$ 2.001/mês a R$ 3.000/mês",
    4: "de R$ 3.001/mês a R$ 4.000/mês",
    5: "de R$ 4.001/mês a R$ 6.000/mês",
    6: "de R$ 6.001/mês a R$ 8.000/mês",
    7: "de R$ 8.001/mês a R$ 12.000/mês",
    8: "de R$ 12.001/mês a R$ 16.000/mês",
    9: "de R$ 16.001/mês a R$ 20.000/mês",
    10: "de R$ 20.001/mês a R$ 25.000/mês",
    11: "de R$ 25.001/mês a R$ 30.000/mês",
    12: "de R$ 30.001/mês a R$ 40.000/mês",
    13: "Acima de R$ 40.001/mês"
}

mapa_faixas = F.create_map(
    *[
        item
        for ordem, faixa in FAIXAS_SALARIAIS.items()
        for item in (
            F.lit(ordem),
            F.lit(faixa)
        )
    ]
)


# ---------------------------------------------------------------------
# FUNÇÃO PARA CALCULAR P50 E P75
"""
A mesma função é aplicada às três dimensões para manter um critério comparável. O P50 corresponde à primeira faixa que alcança 50% da distribuição acumulada e o P75 à primeira que alcança 75%, sempre calculados separadamente por edição e grupo analisado.
"""
"""
O ranking utiliza o P50 como critério principal e o P75 como segundo critério. O dense_rank mantém a mesma posição para grupos com a mesma combinação de faixas. O status de amostra apenas sinaliza grupos com menos de 30 respondentes, sem alterar os cálculos.
"""
# ---------------------------------------------------------------------

def resumo_salario_por_dimensao(
    base,
    dimensao
):
    distribuicao = (
        base
        .groupBy(
            "edicao",
            dimensao,
            "ordem_faixa",
            "valor"
        )
        .agg(
            F.sum("contagem").alias("contagem")
        )
    )

    janela_total = (
        Window
        .partitionBy(
            "edicao",
            dimensao
        )
    )

    janela_acumulada = (
        Window
        .partitionBy(
            "edicao",
            dimensao
        )
        .orderBy("ordem_faixa")
        .rowsBetween(
            Window.unboundedPreceding,
            Window.currentRow
        )
    )

    distribuicao = (
        distribuicao
        .withColumn(
            "total_respondentes",
            F.sum("contagem").over(janela_total)
        )
        .withColumn(
            "acumulado",
            F.sum("contagem").over(janela_acumulada)
        )
        .withColumn(
            "pct_na_dimensao",
            F.round(
                (
                    F.col("contagem")
                    / F.col("total_respondentes")
                ) * 100,
                1
            )
        )
    )

    resumo = (
        distribuicao
        .groupBy(
            "edicao",
            dimensao
        )
        .agg(
            F.max("total_respondentes").alias("total_respondentes"),
            F.min(
                F.when(
                    F.col("acumulado")
                    >= F.col("total_respondentes") * 0.50,
                    F.col("ordem_faixa")
                )
            ).alias("ordem_faixa_mediana"),
            F.min(
                F.when(
                    F.col("acumulado")
                    >= F.col("total_respondentes") * 0.75,
                    F.col("ordem_faixa")
                )
            ).alias("ordem_faixa_p75")
        )
        .withColumn(
            "faixa_salarial_mediana",
            F.element_at(
                mapa_faixas,
                F.col("ordem_faixa_mediana")
            )
        )
        .withColumn(
            "faixa_salarial_p75",
            F.element_at(
                mapa_faixas,
                F.col("ordem_faixa_p75")
            )
        )
        .withColumn(
            "status_amostra",
            F.when(
                F.col("total_respondentes") < 30,
                "Amostra pequena (<30)"
            )
            .otherwise("OK")
        )
    )

    janela_ranking = (
        Window
        .partitionBy("edicao")
        .orderBy(
            F.desc("ordem_faixa_mediana"),
            F.desc("ordem_faixa_p75")
        )
    )

    resumo = (
        resumo
        .withColumn(
            "ranking",
            F.dense_rank().over(janela_ranking)
        )
        .select(
            "edicao",
            dimensao,
            "total_respondentes",
            "faixa_salarial_mediana",
            "faixa_salarial_p75",
            "ordem_faixa_mediana",
            "ordem_faixa_p75",
            "ranking",
            "status_amostra"
        )
        .orderBy(
            "edicao",
            "ranking",
            dimensao
        )
    )

    return resumo


# ---------------------------------------------------------------------
# DIFERENÇAS SALARIAIS POR REGIÃO
"""
Em 2024-2025, o Sudeste apresenta P50 de R$ 8.001 a R$ 12.000, acima das demais regiões, que ficam em R$ 6.001 a R$ 8.000. Em 2025-2026, todas as regiões passam a ter P50 de R$ 8.001 a R$ 12.000; o Norte se diferencia pelo P75 de R$ 16.001 a R$ 20.000, embora seja a menor amostra regional, com 32 respondentes.
"""
# ---------------------------------------------------------------------

print("\n" + "=" * 120)
print("3. DIFERENÇAS SALARIAIS POR REGIÃO")
print("=" * 120)

salario_regiao = resumo_salario_por_dimensao(
    df,
    "regiao_onde_mora"
)

salario_regiao.show(
    100,
    truncate=False
)

exportar_csv(
    salario_regiao,
    "06_01_salario_por_regiao"
)


# ---------------------------------------------------------------------
# DIFERENÇAS SALARIAIS POR SENIORIDADE
"""
A senioridade apresenta uma separação salarial consistente. Júnior permanece com P50 de R$ 3.001 a R$ 4.000, Pleno com R$ 6.001 a R$ 8.000 e Sênior com R$ 12.001 a R$ 16.000 nas duas edições. Especialista/Staff+ aparece em 2025-2026 com P50 de R$ 16.001 a R$ 20.000 e P75 de R$ 20.001 a R$ 25.000.
"""
# ---------------------------------------------------------------------

print("\n" + "=" * 120)
print("4. DIFERENÇAS SALARIAIS POR SENIORIDADE")
print("=" * 120)

salario_senioridade = resumo_salario_por_dimensao(
    df,
    "nivel"
)

salario_senioridade.show(
    100,
    truncate=False
)

exportar_csv(
    salario_senioridade,
    "06_02_salario_por_senioridade"
)


# ---------------------------------------------------------------------
# DIFERENÇAS SALARIAIS POR MODELO DE TRABALHO
"""
Em 2025-2026, remoto, híbrido flexível e híbrido com dias fixos apresentam P50 de R$ 8.001 a R$ 12.000, enquanto o modelo presencial permanece em R$ 4.001 a R$ 6.000. O remoto também apresenta o maior P75 da edição, entre R$ 16.001 e R$ 20.000.
"""
# ---------------------------------------------------------------------

print("\n" + "=" * 120)
print("5. DIFERENÇAS SALARIAIS POR MODELO DE TRABALHO")
print("=" * 120)

salario_modelo = resumo_salario_por_dimensao(
    df,
    "modelo_trabalho"
)

salario_modelo.show(
    100,
    truncate=False
)

exportar_csv(
    salario_modelo,
    "06_03_salario_por_modelo_trabalho"
)